# Análise da Camada Gold (Star Schema)

Este notebook demonstra o uso da **Camada Gold** do Data Warehouse, modelada em um esquema estrela (Star Schema) otimizado para análise (OLAP).

## Objetivos
1. Conectar ao banco de dados `gold_warehouse.duckdb`.
2. Explorar as dimensões e fatos disponíveis.
3. Executar consultas analíticas simplificadas.

In [ ]:
# Importações das bibliotecas necessárias para o notebook
from pathlib import Path

import duckdb
import pandas as pd

In [2]:
# Configurações do ambiente para exibição de dados
pd.options.display.float_format = '{:,.2f}'.format

In [3]:
# Configurações do Data Warehouse e parâmetros de consulta
CONFIG: dict[str, str | Path] = {
    # Use pathlib for construct the path to the DuckDB database
    # This resolves the path relative to the current working directory
    'WAREHOUSE_PATH': Path.cwd() / '..' / 'data' / 'gold_warehouse.duckdb',
    'TARGET_DATE': '2024-12-01',
    'TARGET_INSTITUTION': 'COOPERATIVA DE CRÉDITO CREDIJEQUITINHONHA LTDA. - SICOOB CREDIJEQUITINHONHA',
    'TABLE_SUMMARY': 'prudential_conglomerates_summary',
}

In [ ]:
def get_duckdb_connection(data_warehouse_path: Path) -> duckdb.DuckDBPyConnection | None:
    """Estabelece conexão com o DuckDB com parametros de segurança e performance."""

    try:
        connection = duckdb.connect(
            database=str(data_warehouse_path.resolve()),
            read_only=True,  # PoC de análise deve ser read-only para segurança
            config={'memory_limit': '4GB', 'threads': '4'},
        )
        print(f"Conectado ao Data Warehouse: {data_warehouse_path.name}")
        return connection
    except duckdb.Error as error:
        print(f"Erro ao conectar: {error}")
        return None

In [5]:
# Inicializa a conexão com o Data Warehouse.
duckdb_connection = get_duckdb_connection(CONFIG['WAREHOUSE_PATH'])

Conectado ao Data Warehouse: gold_warehouse.duckdb


## 1. Explorando o Schema

Listando as tabelas disponíveis na camada Gold.

In [6]:
if duckdb_connection:
    duckdb_connection.sql("SHOW TABLES").show()

┌─────────────────────────────┐
│            name             │
│           varchar           │
├─────────────────────────────┤
│ dim_atividade               │
│ dim_classe_instituicao      │
│ dim_consolidado             │
│ dim_controle                │
│ dim_faixa_vencimento        │
│ dim_instituicao             │
│ dim_localizacao             │
│ dim_porte                   │
│ dim_produto_credito         │
│ dim_regiao                  │
│     ·                       │
│     ·                       │
│     ·                       │
│ seed_dim_atividade          │
│ seed_dim_classe_instituicao │
│ seed_dim_consolidado        │
│ seed_dim_controle           │
│ seed_dim_faixa_vencimento   │
│ seed_dim_porte              │
│ seed_dim_produto_credito    │
│ seed_dim_regiao             │
│ seed_dim_risco              │
│ seed_dim_segmento           │
├─────────────────────────────┤
│     36 rows (20 shown)      │
└─────────────────────────────┘



## 2. Consultas Analíticas

Graças ao modelo dimensional, podemos responder perguntas de negócio complexas com SQL simples.

### 2.1 Top 10 Instituições por Ativo Total
Unindo `fato_balanco_patrimonial` com `dim_instituicao`.

In [7]:
query_top_institutuin_ativo_total = """
    SELECT 
        i.nome,
        i.tipo_instituicao,
        t.ano,
        t.trimestre,
        f.depositos,
        f.operacoes_de_credito,
        f.ativo_total
    FROM
        fato_balanco_patrimonial f
    JOIN dim_instituicao i ON f.id_instituicao = i.id_instituicao
    JOIN dim_tempo t ON f.id_data = t.id_data
    WHERE
        t.ano = 2024 AND t.mes = 12
        AND i.tipo_instituicao = 'Conglomerado Financeiro'
    ORDER BY f.ativo_total DESC
    LIMIT 10
"""

In [8]:
if duckdb_connection:
    df_top_10 = duckdb_connection.sql(query_top_institutuin_ativo_total).df()
    display(df_top_10)

,nome,tipo_instituicao,ano,trimestre,depositos,operacoes_de_credito,ativo_total
0,ITAU,Conglomerado Financeiro,2024,4,"1,102,160,071.00","835,635,320.00","2,766,166,428.00"
1,BB,Conglomerado Financeiro,2024,4,"899,076,480.00","987,073,873.00","2,410,321,787.00"
2,CAIXA ECONÔMICA FEDERAL,Conglomerado Financeiro,2024,4,"780,585,537.00","1,218,996,540.00","2,026,456,971.00"
3,BRADESCO,Conglomerado Financeiro,2024,4,"654,882,041.00","598,462,175.00","1,744,567,705.00"
4,SANTANDER BRASIL,Conglomerado Financeiro,2024,4,"495,908,664.00","452,048,377.00","1,369,373,832.00"
5,BANCO NACIONAL DE DESENVOLVIMENTO ECONOMICO E ...,Conglomerado Financeiro,2024,4,"4,885,748.00","296,953,829.00","835,928,013.00"
6,UBS PACTUAL,Conglomerado Financeiro,2024,4,"165,733,228.00","109,666,368.00","649,637,478.00"
7,SAFRA,Conglomerado Financeiro,2024,4,"78,415,839.00","96,733,816.00","302,042,204.00"
8,XP,Conglomerado Financeiro,2024,4,"69,722,476.00","17,493,072.00","301,760,064.00"
9,CITIBANK,Conglomerado Financeiro,2024,4,"81,114,280.00","19,327,595.00","229,633,957.00"


### 2.2 Análise da Carteira de Crédito (Produto x Faixa de Vencimento)

Esta consulta demonstra como pivotar os dados da `fato_carteira_credito_produto` para analisar o perfil da carteira.

In [9]:
query_carteira_credito_produtos = """
    SELECT 
        r.nome_risco as nivel_risco,
        SUM(f.valor) as valor_carteira,
        ROUND(SUM(f.valor) / SUM(SUM(f.valor)) OVER () * 100, 2) as pct
    FROM
        fato_risco_credito f
    JOIN dim_risco r ON f.id_nivel_risco = r.id_risco
    JOIN dim_tempo t ON f.id_data = t.id_data
    WHERE
        t.ano = 2024
            AND t.mes = 12
                AND r.nome_risco <> 'Exterior'
    GROUP BY 1
    ORDER BY valor_carteira DESC
"""

In [10]:
if duckdb_connection:
    df_carteira_credito_produtos = duckdb_connection.sql(query_carteira_credito_produtos).df()
    display(df_carteira_credito_produtos)

,nivel_risco,valor_carteira,pct
0,AA,"2,441,712,062.00",38.11
1,A,"1,973,781,746.00",30.81
2,B,"985,888,130.00",15.39
3,C,"490,041,295.00",7.65
4,H,"182,119,216.00",2.84
5,D,"141,334,726.00",2.21
6,E,"93,847,517.00",1.46
7,G,"49,385,169.00",0.77
8,F,"48,710,437.00",0.76


In [11]:
# Encerra a conexão com o Data Warehouse para liberar recursos.
duckdb_connection.close()